In [1]:
import csv
import pathlib
import numpy as np

from collections import defaultdict
from tqdm import tqdm

1. Split up DO and PO into separate result sets
2. Define coding scheme "anchors" -- fixed values which will be mapped onto the binary code. 
   P.S. The fact that they're fixed does not matter since we will have all possible codes, so every code and its complement will be included.
3. Generate all possible codes
4. Generate versions of the results w/ coding scheme for both DO and PO
   Label for actual HAAP codes
5. save in a gitignored directory (256 x 2 files)

In [42]:
(np.arange(2**8 // 2, dtype=np.uint8)[:, None] >> np.arange(7, -1, -1)) & 1

(128, 8)

In [50]:
# all_codes = (np.arange(2**8, dtype=np.uint8)[:, None] >> np.arange(7, -1, -1)) & 1
all_codes = (np.arange(2**8 // 2, dtype=np.uint8)[:, None] >> np.arange(7, -1, -1)) & 1

HAAP_DO = np.array([0,0,0,0,1,1,1,1])
HAAP_PO = np.array([1,0,1,1,0,1,0,0])

FIXED_ANCHOR = ("pronoun", "animate", "definite", "given", "pronoun", "animate", "definite", "given")

In [51]:
def feature2code(feature,code):
    vector = [0,0,0,0,0,0,0,0]
    for i, (f, fa, c) in enumerate(zip(feature, FIXED_ANCHOR, code)):
        if f == fa:
            vector[i] = c
        else:
            vector[i] = 1-c
    return vector

In [52]:
feature2code(("pronoun", "animate", "definite", "given", "pronoun", "animate", "definite", "given"), HAAP_PO)

[1, 0, 1, 1, 0, 1, 0, 0]

In [53]:
def read_csv_dict(path):
    data = []
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        for line in reader:
            data.append(line)
    return data

def write_csv_dict(data, path):
    with open(path, "w") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        for line in data:
            writer.writerow(line)

In [54]:
# results = read_csv_dict("../data/results/simulation-results/final-results-25-10-16.csv")
adaptation = read_csv_dict("../data/experiments/final-adaptation-25-10-16.csv")

In [55]:
def get_features(entry):
    return entry['theme_pronominality'], entry['theme_animacy'], entry['theme_definiteness'], entry['theme_givenness'], entry['recipient_pronominality'], entry['recipient_animacy'], entry['recipient_definiteness'], entry['recipient_givenness'], int(entry['length_diff'])

In [56]:
idxes = set()
for entry in adaptation:
    idxes.add(entry['idx'])

In [57]:
len(idxes)

12096

In [58]:
'''
id, seed, givennesstemplate, dative, do, pp, [vector], lengthdiff
'''

dative_results = defaultdict(lambda: defaultdict(list))

for entry in tqdm(adaptation):
    dative = entry['dative']
    features = get_features(entry)
    for code_id, code in enumerate(all_codes):
        haaps = {
            'do': False,
            'po': False,
            'do_recipient': False,
            'po_recipient': False,
            'do_theme': False,
            'po_theme': False,
        }
        vector = feature2code(features[:-1], code)
        tp,ta,td,tg,rp,ra,rd,rg = vector
        if sum(code==HAAP_DO) == 8 or sum(code == (1-HAAP_DO)) == 8:
            haaps['do'] = True
        if sum(code==HAAP_PO) == 8 or sum(code==(1-HAAP_PO)) == 8:
            haaps['po'] = True
        if sum(code[:4]==HAAP_DO[:4]) == 4 or sum(code[:4]==(1-HAAP_DO[:4])) == 4:
            haaps['do_theme'] = True
        if sum(code[:4]==HAAP_PO[:4]) == 4 or sum(code[:4]==(1-HAAP_PO[:4])) == 4:
            haaps['po_theme'] = True
        if sum(code[4:]==HAAP_DO[4:]) == 4 or sum(code[4:]==(1-HAAP_DO[4:])) == 4:
            haaps['do_recipient'] = True
        if sum(code[4:]==HAAP_PO[4:]) == 4 or sum(code[4:]==(1-HAAP_PO[4:])) == 4:
            haaps['po_recipient'] = True
            
        dative_results[dative][code_id].append({
            'idx': entry['idx'],
            'item': entry['item'],
            'dative': dative,
            'length_diff': entry['length_diff'],
            "code_id": code_id,
            'code_score': sum(vector),
            'code_score_theme': sum(vector[:4]),
            'code_score_recipient': sum(vector[4:]),
            'haap_do': haaps['do'],
            'haap_po': haaps['po'],
            'haap_do_theme': haaps['do_theme'],
            'haap_po_theme': haaps['po_theme'],
            'haap_do_recipient': haaps['do_recipient'],
            'haap_po_recipient': haaps['po_recipient']
        })
            
#         if dative == 'do':
#             if sum(code == HAAP_DO) == 8:
#                 haap = True
#             else:
#                 haap = False          
                
#         elif dative == 'po':
#             if sum(code == HAAP_PO) == 8:
#                 haap = True
#             else:
#                 haap = False
                
        

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 12096/12096 [00:25<00:00, 470.69it/s]


In [59]:
pathlib.Path("../data/results/simulation-results/haap-25-10-16/").mkdir(parents=True, exist_ok=True)

In [60]:
for dative, code_results in dative_results.items():
    for code_id, res in code_results.items():
        write_csv_dict(res, f"../data/results/simulation-results/haap-25-10-16/{dative}_{code_id}.csv")

In [61]:
len(res)

6048

In [18]:
all_codes[127], all_codes[15]

(array([0, 1, 1, 1, 1, 1, 1, 1]), array([0, 0, 0, 0, 1, 1, 1, 1]))

In [27]:
all_codes[177], all_codes[190], all_codes[65], all_codes[78]

(array([1, 0, 1, 1, 0, 0, 0, 1]),
 array([1, 0, 1, 1, 1, 1, 1, 0]),
 array([0, 1, 0, 0, 0, 0, 0, 1]),
 array([0, 1, 0, 0, 1, 1, 1, 0]))

In [40]:
177-78, 190-65

(99, 125)

In [33]:
all_codes[0], all_codes[112]

(array([0, 0, 0, 0, 0, 0, 0, 0]), array([0, 1, 1, 1, 0, 0, 0, 0]))

In [36]:
180-112

68

In [37]:
all_codes[180], all_codes[68]

(array([1, 0, 1, 1, 0, 1, 0, 0]), array([0, 1, 0, 0, 0, 1, 0, 0]))

In [29]:
177-65, 190-78

(112, 112)

In [22]:
vector

[1, 1, 1, 0, 0, 1, 1, 0]

In [9]:
get_features(results[0])[:-1]

feature2code(get_features(results[0])[:-1], HAAP_DO)

[0, 0, 0, 1, 1, 1, 0, 0]

In [51]:
def haap_coding(entry):
    dative = entry['dative']
    features = get_features(entry)[:-1]
    tp,ta,td,tg,rp,ra,rd,rg = [1,1,1,1,0,0,0,0]
    
    # animacy is globally coded:
    if features[1] == "animate":
        ta = 0
    if features[5] == "animate":
        ra = 1
    # exposure-wise harmonic alignment coding
    if dative == "do":
        if features[0] == "pronoun":
            tp = 0
        if features[2] == "definite":
            td = 0
        if features[3] == "given":
            tg = 0
        if features[4] == "pronoun":
            rp = 1
        if features[6] == "definite":
            rd = 1
        if features[7] == "given":
            rg = 1
            
    elif dative == "pp":
        if features[0] == "noun":
            tp = 0
        if features[2] == "indefinite":
            td = 0
        if features[3] == "new":
            tg = 0
        if features[4] == "noun":
            rp = 1
        if features[6] == "indefinite":
            rd = 1
        if features[7] == "new":
            rg = 1
    return [tp,ta,td,tg,rp,ra,rd,rg]

In [23]:
2**16

65536

In [ ]:
'''
Scheme design:

assign binary code based on fixed positional values:

do: (pronoun, animate, definite, given, pronoun, animate, definite, given)
po: (pronoun, animate, definite, given, pronoun, animate, definite, given)

HAAP:
do: (0,0,0,0,1,1,1,1)
po: (1,0,1,1,0,1,0,0)

or in the full vector form:

'''

In [17]:
def code(entry, scheme):
    '''codes an entry based on the specified'''

[0, 0, 0, 1, 1, 1, 0, 0]

In [18]:
get_features(results[0])

('pronoun',
 'animate',
 'definite',
 'new',
 'pronoun',
 'animate',
 'indefinite',
 'new',
 0)

'\n1. Split up DO and PO into separate result sets\n2. Define coding scheme "anchors" -- fixed values which will be mapped onto the binary code. \n   P.S. The fact that they\'re fixed does not matter since we will have all possible codes, so every code and its complement will be included.\n3. Generate all possible codes\n4. Generate versions of the results w/ coding scheme for both DO and PO\n   Label for actual HAAP codes\n5. save in a gitignored directory (256 x 2 files)\n'